# Reproducing Chandravadia et al. (2020) from dandiset 000004

Paper. Chandravadia N, Liang D, Schjetnan AGP, Carlson A, Faraut M, Chung JM, Reed CM, Dichter B,
Maoz U, Kalia SK, Valiante TA, Mamelak AN, Rutishauser U (2020). A NWB-based dataset and processing
pipeline of human single-neuron activity during a declarative memory task. *Scientific Data* 7, 78.
[doi:10.1038/s41597-020-0415-9](https://doi.org/10.1038/s41597-020-0415-9). The claims file was written
from the open access full text (CC BY 4.0), its Methods, Data Records, Technical Validation, figure
legends, Tables 1 and 2 and Online-only Table 1, and from the paper's code
(github.com/rutishauserlab/recogmem-release-NWB at commit 1d95162).

Dandiset. [DANDI:000004](https://dandiarchive.org/dandiset/000004/0.220126.1852), version
0.220126.1852 (doi:10.48324/dandi.000004/0.220126.1852). This version was asked for in the task that
started this attempt. The paper cites no DANDI version (its data citation is the OSF project
doi:10.17605/OSF.IO/HV7JA, which the dandiset names as its source). The archive has two published
versions, 0.220126.1851 and 0.220126.1852, and a draft; the cell "Versions" below compares their files.

Outside inputs. Online-only Table 1 of the paper and Table 1 of Faraut et al. 2018 (Sci Data 5,
180010, doi:10.1038/sdata.2018.10, the earlier release of this task's data) are read at run time from
their full texts in Europe PMC (PMC7055261 and PMC5810422). The first is the paper's own list of
subjects and sessions, the reference the claims compare with; the second is used only by the two
claims about the earlier release (`previous-release-link`, `previous-release-additions`).

Method choices fixed by the claims file (its opening comment) are in the docstring of `nomem.py`:
recognition trials are the rows with `stim_phase` "recog", the confidence is `response_value` minus 30,
"old" is label 1 (as the paper's code reads it), firing rates are spike counts in a window over its
length, the Methods' window 0.2 to 1.2 s after stimulus onset is the main one and the code's 0.2 to
1.7 s a variant, and the behavioural measures follow the paper's code.

| Claim | Tested here |
|---|---|
| counts-neurons-subjects | tested |
| counts-sessions | tested |
| table1-demographics | tested |
| task-structure | tested |
| new-old-label-coding | tested |
| ttl-codes | tested |
| experiment-ids | tested |
| session-dates | tested |
| units-table | tested |
| electrodes-table | tested |
| isi-criterion | tested |
| fig5a-mtl-locations | tested |
| fig5c-auc-all | tested |
| fig5c-auc-sites | tested |
| fig5b-roc-asymmetric | tested |
| fig5d-confidence-accuracy | tested |
| vs-cells-proportion | tested |
| ms-cells-proportion | tested |
| dual-cells-proportion | tested |
| fig5f-site-proportions | tested |
| twh-sessions | tested |
| previous-release-link | tested, with Faraut et al. 2018 Table 1 |
| previous-release-additions | tested, with Faraut et al. 2018 Table 1 |
| fig4-example-vs-neuron | tested |
| fig3-example-vs-neuron | blocked: session 144 is not in the dandiset |

In [ ]:
import collections
import hashlib
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dandi.dandiapi import DandiAPIClient
from scipy import stats

import nomem

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=np.exceptions.RankWarning)

started = time.time()
sessions = nomem.load_all()
print(f"loaded {len(sessions)} files from dandiset {nomem.DANDISET_ID} version {nomem.DANDISET_VERSION} "
      f"in {time.time() - started:.0f} s")
paper_t1, paper_totals = nomem.paper_table1()
print("Online-only Table 1:", len(paper_t1), "subject rows; totals row:", paper_totals)

## Data check

Each file is one session. Its identifier (`H09_5`) gives the subject label of Online-only Table 1 and
the session id ("NOID"); Cedars-Sinai labels are written `CS24` in the identifiers and `C24` in the
table, and are matched as the same label. The paper's variant of a session is read from its category
names (Faraut et al. 2018, Table 2: variant 1 houses, landscapes, mobility, phones, animals; variant 2
fruit, kids, military, space, animals; variant 3 cars, food, people, spatial, animals).

In [ ]:
VARIANT_OF_CATEGORY = {"houses": 1, "fruit": 2, "1cars": 3}


def norm_label(label):
    return "C" + label[2:] if label.startswith("CS") else label


def variant(s):
    names = set(s["trials"].category_name)
    found = [v for k, v in VARIANT_OF_CATEGORY.items() if k in names]
    return found[0] if len(found) == 1 else np.nan


def learn_recog_ids(s):
    # the experiment id whose events hold the encoding answers (20, 21) is the learning block
    ev, ex = s["events"], s["expids"]
    learn = sorted(set(ex[np.isin(ev, [20, 21])]))
    recog = sorted(set(ex[(ev >= 31) & (ev <= 36)]))
    return learn, recog


rows = []
for s in sessions:
    label, sid = nomem.session_key(s)
    t = s["trials"]
    r = t[t.stim_phase == "recog"]
    lid, rid = learn_recog_ids(s)
    rows.append({"file": s["path"].split("/")[-1][:40], "identifier": s["identifier"], "label": norm_label(label),
                 "session": sid, "site": nomem.site(label), "subject_id": s["subject"]["subject_id"],
                 "age": s["subject"]["age"], "sex": s["subject"]["sex"], "start": s["session_start_time"][:10],
                 "n_learn": int((t.stim_phase == "learn").sum()), "n_recog": len(r),
                 "label0": int((r.new_old_labels_recog == "0").sum()), "label1": int((r.new_old_labels_recog == "1").sum()),
                 "n_cat": t.stimCategory.nunique(), "variant": variant(s), "units": len(s["units"]),
                 "electrodes": len(s["electrodes"]), "learn_id": lid, "recog_id": rid,
                 "institution": s["institution"], "device": ",".join(s["devices"])})
info = pd.DataFrame(rows).sort_values(["site", "label", "session"]).reset_index(drop=True)
print("Files:", len(info), "; distinct subject_id:", info.subject_id.nunique(), "; distinct labels:", info.label.nunique())
print("Files by site:", info.site.value_counts().to_dict())
print("Unit rows:", info.units.sum(), "; electrode rows:", info.electrodes.sum())
print("Trial phases over all files:", collections.Counter(x for s in sessions for x in s["trials"].stim_phase))
print("Institution field:", collections.Counter(info.institution))
print("Devices:", collections.Counter(info.device))
print(info.drop(columns=["file"]).to_string())

The subject ids of the files (`P9HMH`, `P24CS`, `TWH088`) are a third spelling of the subject labels;
each subject id goes with one label. The institution field is spelled two ways for Huntington
Memorial Hospital ("Hunigton" in 28 files, "Hunington" in 6), neither correct.

In [ ]:
# Versions: the files of the two published versions and the draft, by path, size and blob.
with DandiAPIClient() as client:
    listings = {}
    for v in ["0.220126.1851", nomem.DANDISET_VERSION, "draft"]:
        d = client.get_dandiset(nomem.DANDISET_ID, v)
        listings[v] = {a.path: a.blob for a in d.get_assets()}
pinned = listings[nomem.DANDISET_VERSION]
for v, files in listings.items():
    same = sum(1 for p in pinned if files.get(p) == pinned[p])
    print(f"{v}: {len(files)} files; {same} with the pinned version's path and blob (content); "
          f"paths only in one of the two: {sorted(set(files) ^ set(pinned))[:5]}")

Files are compared by path and by the archive's blob id, which changes when the content changes.

In [ ]:
# Units and ranges of every variable used.
spk = np.concatenate([np.concatenate(s["spikes"]) for s in sessions])
units_all = pd.concat([s["units"].assign(identifier=s["identifier"]) for s in sessions], ignore_index=True)
print("spike_times attributes (description, unit, resolution) over all files:",
      collections.Counter(tuple(sorted(s["spike_times_attrs"].items())) for s in sessions))
print("Unit of the events timestamps:", collections.Counter(s["event_time_unit"] for s in sessions))
print("Spike times: range per file from", f"{min(np.concatenate(s['spikes']).min() for s in sessions):.1f}",
      "to", f"{max(np.concatenate(s['spikes']).max() for s in sessions):.4g} s")
starts = info.set_index("identifier")
epoch = [s["identifier"] for s in sessions if s["trials"].start_time.min() > 1e8]
print(f"Files whose trial and spike times are Unix time (> 1e8 s): {len(epoch)}; the others count from an "
      "arbitrary origin")
dur = [(s["trials"].stop_time - s["trials"].start_time).median() for s in sessions]
print(f"Median trial duration (start_time to stop_time) per file: {min(dur):.2f} to {max(dur):.2f} s")
on_off = [(s["trials"].stim_off_time - s["trials"].stim_on_time).median() for s in sessions]
print(f"Median stimulus duration (stim_off - stim_on): {min(on_off):.3f} to {max(on_off):.3f} s (Faraut 2018: 1 s)")
print("start_time equals stim_on_time in every file:",
      all(np.allclose(s["trials"].start_time, s["trials"].stim_on_time) for s in sessions))
rv = pd.concat([s["trials"][["stim_phase", "response_value"]] for s in sessions])
print("response_value by phase:", {k: sorted(set(v.astype(int))) for k, v in rv.groupby("stim_phase").response_value})
print("Waveform sampling rate (Hz):", sorted(set(units_all.waveform_rate)), "; samples per waveform:",
      sorted(set(units_all.waveform_mean_encoding_samples)))
print("SNR range:", np.nanmin(units_all.SNR).round(2), "to", np.nanmax(units_all.SNR).round(2),
      "; IsolationDist range:", np.nanmin(units_all.IsolationDist).round(2), "to", np.nanmax(units_all.IsolationDist).round(1))
print("Units with spike times not in increasing order:",
      sum(int((np.diff(x) < 0).any()) for s in sessions for x in s["spikes"]))
print("Label column description (all files):", collections.Counter(s["label_description"] for s in sessions))

In the recognition phase `response_value` holds the event code (31 to 36), so the confidence is
`response_value` minus 30, as the claims file says. In the encoding phase it holds 0 or 1, not the 20
and 21 of the events. Trial and spike times share one clock in each file (checked below); 38 files
count it in Unix seconds, the others from an arbitrary origin, so only differences of times are
meaningful. The spike_times column states no unit (its description is "the spike times for
each unit"); the NWB schema defines spike times in seconds, and the events timestamps state "seconds".
The median stimulus duration is 1 s in most files and up to 2 s in some. Spike times are
not in increasing order in 13 units (see isi-criterion). SNR is infinite for some units and isolation
distance reaches 1e32 (see units-table). The confidence read from `response_value` is checked against
the event codes the paper's code reads:

In [ ]:
# The confidence from the trials table against the code read by the paper's code: the event three
# after each stimulus onset (1) in the recognition block (NWBloadDataOfBlock_release.m).
mismatch = {}
for s in sessions:
    ev, ex = s["events"], s["expids"]
    _, rid = learn_recog_ids(s)
    block = ev[np.isin(ex, rid)]
    on = np.where(block == 1)[0]
    from_events = block[on + 3] - 30
    from_table = nomem.recog(s).confidence.to_numpy()
    if len(from_events) != len(from_table) or (from_events != from_table).any():
        mismatch[s["identifier"]] = int((from_events != from_table).sum()) if len(from_events) == len(from_table) else "length"
print("Files where the confidence from response_value differs from the events:", mismatch or "none")

In 4 files the confidence from the trials table differs from the code the paper's code reads for 1 to
6 trials; these files hold extra event codes (22, 23 or 0, see ttl-codes) that shift the position the
code reads. The analyses use the trials table, as the claims file says.

In [ ]:
# Spike trains deposited in more than one file (identical sorted spike times).
owners = collections.defaultdict(list)
for s in sessions:
    for i, sp in enumerate(s["spikes"]):
        owners[hashlib.md5(np.sort(sp).tobytes()).hexdigest()].append((s["identifier"], i))
dups = [v for v in owners.values() if len(v) > 1]
print("Spike trains found in two files:", len(dups), "; in more than two:", sum(len(v) > 2 for v in dups))
print("By pair of files:", dict(collections.Counter(tuple(sorted({x[0] for x in v})) for v in dups)))
duplicate_second = {v[1] for v in dups}      # (identifier, unit) of the second copy, left out when counted once
print("Unit rows:", len(units_all), "; with each duplicated spike train counted once:", len(units_all) - len(duplicate_second))
# Do these pairs also share trials?
byid = {s["identifier"]: s for s in sessions}
for a, b in sorted({tuple(sorted({x[0] for x in v})) for v in dups}):
    ta, tb = byid[a]["trials"], byid[b]["trials"]
    la, lb = ta[ta.stim_phase == "learn"], tb[tb.stim_phase == "learn"]
    same_learn = len(la) == len(lb) and np.allclose(la.start_time, lb.start_time)
    ra, rb = ta[ta.stim_phase == "recog"], tb[tb.stim_phase == "recog"]
    print(f"  {a} / {b}: same encoding trial times {same_learn}; same recognition trial times "
          f"{len(ra) == len(rb) and np.allclose(ra.start_time, rb.start_time)}; spike-train range "
          f"{np.concatenate(byid[a]['spikes']).min():.0f} to {np.concatenate(byid[a]['spikes']).max():.0f} s; "
          f"trials of {a} {ta.start_time.min():.0f} to {ta.stop_time.max():.0f} s, of {b} {tb.start_time.min():.0f} to {tb.stop_time.max():.0f} s")

Three pairs of files hold the same spike trains: sessions 38 and 39 (11 units), 43 and 44 (23) and 54
and 55 (32), each pair recorded on the same day from one patient. The spike trains span both sessions'
trials, so each file holds the units of the whole recording. Sessions 54 and 55 also share the encoding
trials: the same learning block was followed by two recognition blocks. Counts with each duplicated
spike train counted once are given as a variant where the claims ask for them.

In [ ]:
# Clock check: are the spike times on the clock of the trial table? For each unit, the rate 0.2 to 1.2 s
# after stimulus onset against the rate 1 s before it (paired t-test over the 200 trials), with the true
# onsets and with every onset moved 37 s later, which keeps the trial structure but misaligns it.
# A second contrast, between conditions: the one-way ANOVA over the 5 categories of the recognition
# trials, with the window moved by a lag from -3 to +3 s; it can only peak near 0 if the clocks agree.
rows = []
for s in sessions:
    on = s["trials"].start_time.to_numpy()
    rec = nomem.recog(s)
    for shift in (0.0, 37.0):
        resp = 0
        for sp in s["spikes"]:
            post = nomem.window_rates(sp, on + shift, (0.2, 1.2))
            pre = nomem.window_rates(sp, on + shift, (-1.0, 0.0))
            if np.any(post != pre) and stats.ttest_rel(post, pre).pvalue < 0.05:
                resp += 1
        rows.append({"identifier": s["identifier"], "shift": shift, "responsive": resp, "units": len(s["spikes"])})
clock = pd.DataFrame(rows).pivot_table(index=["identifier", "units"], columns="shift", values="responsive").reset_index()
clock.columns = ["identifier", "units", "responsive_true", "responsive_shifted"]
print("Units with an evoked response (p < 0.05), all files: true onsets",
      int(clock.responsive_true.sum()), "of", int(clock.units.sum()), "; onsets + 37 s", int(clock.responsive_shifted.sum()))
weak = clock[clock.responsive_true <= clock.responsive_shifted]
print("Files where the true onsets give no more responsive units than the shifted ones:")
print(weak.to_string(index=False))
lags = np.arange(-3.0, 3.01, 0.5)
lag_counts = []
for lag in lags:
    n = 0
    for s in sessions:
        rec = nomem.recog(s)
        for sp in s["spikes"]:
            p = nomem.anova_p(nomem.window_rates(sp, rec.onset.to_numpy() + lag, (0.2, 1.2)), rec.category.to_numpy())
            n += int(p < 0.05) if np.isfinite(p) else 0
    lag_counts.append(n)
print("Units with a category effect (ANOVA p < 0.05) by lag of the window (s):")
print(dict(zip(lags.round(1).tolist(), lag_counts)))

Over all files, 830 of 1864 units respond to the stimulus at the true onsets against 115 at onsets
moved by 37 s, and the number of units with a category effect peaks at lag 0 (332) among lags from -3
to +3 s. Both are consistent with spikes and trials sharing a clock. In the five files listed above (1
to 19 units each) the check cannot establish it on its own.

## counts-neurons-subjects

> The dataset holds 1863 single neurons recorded from the medial temporal lobes of 59 subjects.

Criterion: neurons are the rows of the units tables over all files, subjects the distinct
`subject_id` values; readings (a) every unit row, (b) duplicated spike trains counted once.
Reproduced when both values are within tolerance (1863 ± 186.3, 59 ± 5.9) under one reading.

In [ ]:
n_rows = len(units_all)
n_once = n_rows - len(duplicate_second)
n_subjects = info.subject_id.nunique()
print(f"(a) unit rows: {n_rows} (paper 1863, difference {n_rows - 1863:+d}); within 1863 +- 186.3: {abs(n_rows - 1863) <= 186.3}")
print(f"(b) duplicated spike trains counted once: {n_once} (difference {n_once - 1863:+d}); within: {abs(n_once - 1863) <= 186.3}")
print(f"distinct subject_id: {n_subjects} (paper 59); within 59 +- 5.9: {abs(n_subjects - 59) <= 5.9}")
print("Unit rows by site:", info.groupby("site").units.sum().to_dict())

**Result: Reproduced.** The units tables hold 1864 rows, one more than the paper's 1863, and the files
come from 59 distinct subjects. Counting each of the 66 duplicated spike trains once gives 1798 units,
also within tolerance.

## counts-sessions

> The patients performed 87 sessions, listed by subject and session id in Online-only Table 1 (the Methods give 89).

Criterion: reproduced when the set of (subject, session id) pairs of the files equals the set of
Online-only Table 1; partially when the count is within tolerance (87 ± 8.7) but the sets differ.

In [ ]:
files_pairs = set(zip(info.label, info.session))
table_sessions = nomem.sessions_of(paper_t1)
table_pairs = set(zip(table_sessions.label, table_sessions.session))
print("Files:", len(info), "; distinct (subject, session) pairs in the files:", len(files_pairs))
print("Pairs in Online-only Table 1:", len(table_pairs), "; its stated total:", [t for t in paper_totals if "Sessions" in t])
print("In the files, not the table:", sorted(files_pairs - table_pairs))
print("In the table, not the files:", sorted(table_pairs - files_pairs))
print("Sets equal:", files_pairs == table_pairs, "; the Methods' 89 differs from the file count by", len(info) - 89)

**Result: Reproduced.** The 87 files carry 87 distinct (subject, session) pairs, the same set as
Online-only Table 1. The Methods' 89 is not the number of sessions in the files.

## table1-demographics

> Online-only Table 1 gives the age and sex of every subject; there are 25 female and 34 male subjects.

Criterion: the age and sex in the files are compared with the table subject by subject; reproduced
when every subject matches, partially when some do not, not reproduced when fewer than half match.
The diagnosis is shown side by side.

In [ ]:
subj = info.drop_duplicates("label").set_index("label")
diag = {s["identifier"]: s["subject"].get("description", "") for s in sessions}
subj["diagnosis_file"] = [diag[i] for i in subj.identifier]
cmp = paper_t1.set_index("label")[["age", "sex", "diagnosis"]].join(
    subj[["age", "sex", "diagnosis_file"]].rename(columns={"age": "age_file", "sex": "sex_file"}), how="outer")
cmp["age_file"] = pd.to_numeric(cmp.age_file)
cmp["age_match"] = cmp.age == cmp.age_file
cmp["sex_match"] = cmp.sex == cmp.sex_file
cmp["both_match"] = cmp.age_match & cmp.sex_match
print("Subjects in the table:", len(paper_t1), "; in the files:", len(subj), "; in both:", int(cmp.age.notna().sum()))
print("Age and sex both match:", int(cmp.both_match.sum()), "of", len(cmp))
print("Female / male in the table:", (paper_t1.sex == "F").sum(), "/", (paper_t1.sex == "M").sum(),
      "; in the files:", (subj.sex == "F").sum(), "/", (subj.sex == "M").sum())
print("\nSubjects that differ:")
print(cmp[~cmp.both_match].to_string())
print("\nDiagnosis, table against file (first 15 subjects):")
print(cmp[["diagnosis", "diagnosis_file"]].head(15).to_string())
print("Diagnosis text identical:", int((cmp.diagnosis.str.lower() == cmp.diagnosis_file.str.lower()).sum()), "of", len(cmp))

**Result: Partially reproduced.** Age and sex match Online-only Table 1 for 56 of 59 subjects. C24 is
57 in the files and 47 in the table, H33 30 and 29, and H48 is a 32-year-old woman in the files and a
54-year-old man in the table, so the files hold 26 female and 33 male subjects against the table's 25
and 34. The files' diagnosis text equals the table's for 40 of 59 subjects; the others use coarser
terms ("Right Other" for "Right Insula").

## task-structure

> Each session has 100 encoding trials showing 100 novel images and 100 recognition trials, of which 50 show new and 50 old images, and the images come from 5 visual categories.

Criterion: per session, from the trials table and the stimulus images; reproduced when every session
has 100 encoding trials with 100 distinct image files, 100 recognition trials with 50 of each label,
and 5 categories; partially when each part holds in more than half of the sessions.

In [ ]:
rows = []
for s in sessions:
    t = s["trials"]
    learn = (t.stim_phase == "learn").to_numpy()
    h = np.array(s["image_hash"])
    r = t[~learn]
    rows.append({"identifier": s["identifier"], "encoding_trials": int(learn.sum()),
                 "distinct_files": t.external_image_file[learn].nunique(), "distinct_images": len(set(h[learn])),
                 "recognition_trials": len(r), "label0": int((r.new_old_labels_recog == "0").sum()),
                 "label1": int((r.new_old_labels_recog == "1").sum()), "categories": t.stimCategory.nunique(),
                 "image_frames": s["image_shape"][0]})
ts = pd.DataFrame(rows)
parts = {"100 encoding trials, 100 distinct files": (ts.encoding_trials == 100) & (ts.distinct_files == 100),
         "100 recognition trials, 50 of each label": (ts.recognition_trials == 100) & (ts.label0 == 50) & (ts.label1 == 50),
         "5 categories": ts.categories == 5}
for k, v in parts.items():
    print(f"{k}: {int(v.sum())} of {len(ts)} sessions")
print("Distinct image files equal distinct image contents in every session:", bool((ts.distinct_files == ts.distinct_images).all()))
print("One stored image per trial in every session:", bool((ts.image_frames == ts.encoding_trials + ts.recognition_trials).all()))
print("\nSessions that differ:")
print(ts[~parts["100 encoding trials, 100 distinct files"]].to_string(index=False))

**Result: Partially reproduced.** Every session has 100 recognition trials with 50 of each label and 5
categories, but 8 of 87 sessions have fewer encoding trials: 7 have 50 (sessions 18, 20, 21, 23, 26,
27 and 44, all from Huntington) and one (session 123) has 97. The other 79 have 100 encoding trials
with 100 distinct images.

## new-old-label-coding

> In the trials table, new_old_labels_recog is 0 for old stimuli (seen during encoding) and 1 for new stimuli.

Criterion: (1) under the stated coding, old trials (label 0) show images of the session's encoding
block and new trials (label 1) do not; (2) old trials receive "old" responses (4 to 6) more often than
new trials. Reproduced when both hold in every session; not reproduced when, in most sessions, both
hold with the labels reversed. Images are compared by their stored pixels (a hash per frame) and by
file name.

In [ ]:
rows = []
for s in sessions:
    t = s["trials"]
    learn = (t.stim_phase == "learn").to_numpy()
    h, fn = np.array(s["image_hash"]), t.external_image_file.to_numpy()
    rec = ~learn
    lab = t.new_old_labels_recog.to_numpy()[rec]
    in_enc = np.isin(h[rec], h[learn])
    in_enc_name = np.isin(fn[rec], fn[learn])
    conf = nomem.recog(s).confidence.to_numpy()
    rows.append({"identifier": s["identifier"], "encoding_trials": int(learn.sum()),
                 "label0_in_encoding": in_enc[lab == "0"].mean(), "label1_in_encoding": in_enc[lab == "1"].mean(),
                 "names_agree": bool((in_enc == in_enc_name).all()),
                 "old_resp_label0": (conf[lab == "0"] >= 4).mean(), "old_resp_label1": (conf[lab == "1"] >= 4).mean()})
lc = pd.DataFrame(rows)
stated_1 = (lc.label0_in_encoding == 1) & (lc.label1_in_encoding == 0)
reversed_1 = (lc.label1_in_encoding == 1) & (lc.label0_in_encoding == 0)
reversed_1_partial = (lc.label1_in_encoding > 0.5) & (lc.label0_in_encoding == 0)
stated_2 = lc.old_resp_label0 > lc.old_resp_label1
reversed_2 = lc.old_resp_label1 > lc.old_resp_label0
print("Test 1, stated coding (label 0 images all from encoding, label 1 none):", int(stated_1.sum()), "of", len(lc))
print("Test 1, reversed coding (label 1 images all from encoding, label 0 none):", int(reversed_1.sum()), "of", len(lc))
print("   label 0 images found in encoding block, over all sessions:", lc.label0_in_encoding.max())
print("   reversed, with label 1 mostly (> 50%) from the encoding block:", int(reversed_1_partial.sum()), "of", len(lc))
print("Test 2, stated coding (more 'old' responses to label 0):", int(stated_2.sum()), "of", len(lc))
print("Test 2, reversed coding (more 'old' responses to label 1):", int(reversed_2.sum()), "of", len(lc))
print("Image names and pixels agree on every trial:", bool(lc.names_agree.all()))
print("Both tests hold with the labels reversed, test 1 strict:", int((reversed_1 & reversed_2).sum()), "of", len(lc))
print("Both tests hold with the labels reversed, test 1 with label 1 mostly from encoding:",
      int((reversed_1_partial & reversed_2).sum()), "of", len(lc))
print("\nSessions not fully consistent with the reversed coding:")
print(lc[~(reversed_1 & reversed_2)].round(2).to_string(index=False))
print("\nMean 'old' response rate: label 0", lc.old_resp_label0.mean().round(3), "; label 1", lc.old_resp_label1.mean().round(3))

In [ ]:
# Where are the other 'old' images of the sessions with 50 encoding trials? Sessions of one variant show
# the same images; look for each old image among the encoding images of all files.
enc_sets = {s["identifier"]: set(np.array(s["image_hash"])[(s["trials"].stim_phase == "learn").to_numpy()]) for s in sessions}
for s in sessions:
    t = s["trials"]
    if (t.stim_phase == "learn").sum() == 100:
        continue
    h = np.array(s["image_hash"])
    old = set(h[((t.stim_phase == "recog") & (t.new_old_labels_recog == "1")).to_numpy()])
    anywhere = set().union(*[v for k, v in enc_sets.items()])
    print(f"{s['identifier']}: {len(old & enc_sets[s['identifier']])} of {len(old)} old images in its own encoding "
          f"block; {len(old & anywhere)} in the encoding block of some file")

**Result: Not reproduced.** Under the stated coding test 1 holds in 0 of 87 sessions and test 2 in 1.
With the labels reversed, test 1 holds as stated (all label-1 images from the encoding block, no
label-0 image) in 79 sessions and test 2 in 86, and both in 78 of 87. No label-0 image is in the
encoding block of its session in any of the 87; the 8 sessions that miss test 1 do so because their
files lack encoding trials (49 of 50 label-1 images found in session 123, 26 to 29 of 50 in the seven
with 50 encoding trials). Counting a session when most of its label-1 images are from its encoding
block, a threshold of this attempt, both tests hold in 86. Label-1 trials draw more "old" responses
(69% against 27% on average); the exception for test 2 is session 2005 (T98, AUC 0.445). The labels are 1
for old and 0 for new, as the paper's code reads them and the reverse of the paper's text and the
column's description. In the seven sessions with 50 encoding trials, 21 to 24 old images are not in
the file's encoding block but are in the encoding blocks of other files of the same variant, so these
files lack the encoding trials that showed them.

## ttl-codes

> The events stream holds the TTL markers 55, 1, 2, 3, 20 or 21, 31 to 36, 6 and 66 with the meanings of Table 1, and the events and experiment_ids streams have one entry each, in the same order.

Criterion: reproduced when, in every file, events and experiment_ids have the same length and
timestamps, the events hold only the listed codes, and every trial shows stimulus onset (1), offset
(2), question (3), a response of the phase's kind and end of trial (6) in this order; partially when
this holds in most files.

In [ ]:
LISTED = {55, 1, 2, 3, 20, 21, 31, 32, 33, 34, 35, 36, 6, 66}
rows = []
for s in sessions:
    ev, ex = s["events"], s["expids"]
    lid, rid = learn_recog_ids(s)
    same = len(ev) == len(ex) and np.array_equal(s["events_t"], s["expids_t"])
    extra = sorted(set(ev.astype(int)) - LISTED)
    ok_trials, n_on = 0, 0
    for ids, kind in ((lid, {20, 21}), (rid, set(range(31, 37)))):
        block = ev[np.isin(ex, ids)].astype(int)
        for i in np.where(block == 1)[0]:
            n_on += 1
            seq = block[i:i + 5]
            ok_trials += int(len(seq) == 5 and seq[1] == 2 and seq[2] == 3 and seq[3] in kind and seq[4] == 6)
    rows.append({"identifier": s["identifier"], "same_length_and_times": same, "extra_codes": extra,
                 "has_55": 55 in ev, "has_66": 66 in ev, "stim_onsets": n_on, "trials": len(s["trials"]),
                 "trials_in_order": ok_trials, "times_increasing": bool(np.all(np.diff(s["events_t"]) >= 0))})
tt = pd.DataFrame(rows)
tt["all_ok"] = tt.same_length_and_times & (tt.extra_codes.str.len() == 0) & (tt.trials_in_order == tt.trials) & (tt.stim_onsets == tt.trials)
print("Same length and timestamps:", int(tt.same_length_and_times.sum()), "of", len(tt))
print("Only listed codes:", int((tt.extra_codes.str.len() == 0).sum()), "of", len(tt), "; extra codes found:",
      dict(collections.Counter(c for x in tt.extra_codes for c in x)))
print("Every trial in order 1, 2, 3, response, 6:", int((tt.trials_in_order == tt.trials).sum()), "of", len(tt),
      "; trials in order over all files:", int(tt.trials_in_order.sum()), "of", int(tt.trials.sum()))
print("Files without 55 (start):", tt[~tt.has_55].identifier.tolist(), "; without 66 (end):", tt[~tt.has_66].identifier.tolist())
print("All parts hold:", int(tt.all_ok.sum()), "of", len(tt))
print(tt[~tt.all_ok].to_string(index=False))

**Result: Partially reproduced.** In all 87 files events and experiment_ids have the same length and
timestamps. 9 files hold codes Table 1 does not list (0, 4, 22, 23, 98, 99), and in 7 of them 1 to 10
trials break the order 1, 2, 3, response, 6 (17019 of 17047 trials follow it), so all parts hold in
78 of 87 files. Seven files have no start marker (55).

## experiment-ids

> The learning block is labelled with one of 80, 83 or 88 and the recognition block with one of 81, 84 or 89.

Criterion: reproduced when the two experiment ids of every file fall in these sets; partially when
they do in most files. The learning block is the id whose events hold the encoding answers (20, 21).

In [ ]:
ex = info[["identifier", "learn_id", "recog_id"]].copy()
ex["learn_ok"] = ex.learn_id.apply(lambda v: len(v) == 1 and v[0] in (80, 83, 88))
ex["recog_ok"] = ex.recog_id.apply(lambda v: len(v) == 1 and v[0] in (81, 84, 89))
ex["pair_ok"] = ex.apply(lambda r: len(r.learn_id) == 1 and len(r.recog_id) == 1 and (r.learn_id[0], r.recog_id[0]) in ((80, 81), (83, 84), (88, 89)), axis=1)
print("Learning id in {80, 83, 88}:", int(ex.learn_ok.sum()), "of", len(ex), "; recognition id in {81, 84, 89}:", int(ex.recog_ok.sum()))
print("Both in the sets:", int((ex.learn_ok & ex.recog_ok).sum()), "; pair one of Table 2's:", int(ex.pair_ok.sum()))
print("Pairs:", dict(collections.Counter((r.learn_id[0], r.recog_id[0]) for r in ex.itertuples())))
desc = {s["identifier"]: s["data_collection"] for s in sessions}
bad = ex[~(ex.learn_ok & ex.recog_ok)].assign(data_collection=lambda d: d.identifier.map(desc))
print(bad.to_string(index=False))

**Result: Partially reproduced.** In 83 of 87 files the two ids are in Table 2's sets; four files use
82 for a block (pairs 80/82, 81/82 and 82/83), which Table 2 does not list. The file's own description
(`data_collection`) gives the same ids.

## session-dates

> To protect patient information, session_start_time keeps only the year and month of the experiment, with the day set to the first of the month for all sessions.

Criterion: reproduced when every file's session_start_time falls on the first of a month; partially
when some do not; years outside the span of the recordings are reported.

In [ ]:
dates = pd.to_datetime(info.start.values)
info["day"], info["year"] = dates.day, dates.year
print("On the first of a month:", int((info.day == 1).sum()), "of", len(info))
print("Not on the first:", info[info.day != 1][["identifier", "start"]].to_string(index=False))
print("Years by site:", info.groupby("site").year.agg(["min", "max"]).to_dict())
print("Years of the H subjects in label order:")
h = info[info.label.str.startswith("H")].assign(n=lambda d: d.label.str[1:].astype(int)).sort_values("n")
print(h[["label", "session", "start"]].drop_duplicates("label").to_string(index=False))

**Result: Partially reproduced.** 86 of 87 start times fall on the first of a month; session 48
(H28) is dated 2010-03-13. Session 68 (H44) is dated 2000-01-01, years before every other Huntington
session (2006 to 2016) and out of order with its neighbours (H43 in 2013, H47 in 2014), so its year is
probably a placeholder.

## units-table

> The units table holds spike_times, waveform_mean_encoding, waveform_mean_recognition (sampled at 100 kHz), origClusterID, SNR, IsolDist and electrodes, with one electrode per neuron.

Criterion: reproduced when every file holds each column, each unit links to exactly one electrode, the
stated sampling rate of the waveforms is within 100 kHz ± 10 kHz, and the quality metrics are finite for
every unit; partially when the columns are there but some values are missing or not finite.

In [ ]:
wanted = ["spike_times", "waveform_mean_encoding", "waveform_mean_recognition", "origClusterID", "SNR", "IsolDist", "electrodes"]
cols = collections.Counter(c for s in sessions for c in s["unit_columns"])
print("Columns over all files:", dict(cols))
print("Missing by name in every file:", [c for c in wanted if cols[c] == 0])
print("Column descriptions over all files:")
for c in sorted({c for s in sessions for c in s["unit_descriptions"]}):
    print(f"  {c}:", dict(collections.Counter(s["unit_descriptions"].get(c) for s in sessions)))
print("Electrodes per unit:", dict(collections.Counter(len(x) for s in sessions for x in s["unit_electrodes"])))
rate = sorted(set(units_all.waveform_rate))
print("Waveform sampling rate:", rate, "Hz; within 100000 +- 10000:", all(abs(r - 1e5) <= 1e4 for r in rate))
print("Waveforms finite for every unit:", bool(units_all.waveform_mean_encoding_finite.all() and units_all.waveform_mean_recognition_finite.all()))
for c in ("SNR", "IsolationDist", "origClusterID"):
    v = units_all[c].astype(float)
    print(f"{c}: not finite for {int((~np.isfinite(v)).sum())} of {len(v)} units (NaN {int(v.isna().sum())}, "
          f"infinite {int(np.isinf(v).sum())}), in {units_all[~np.isfinite(v)].identifier.nunique()} files; above 1e6: {int((v > 1e6).sum())}")

**Result: Partially reproduced.** Every file holds the columns (IsolDist under the name
`IsolationDist`, described as "IsolDist"), each of the 1864 units links to one electrode, and the
waveforms are stated as sampled at 98.4 kHz, within tolerance. SNR is not finite for 86 units (NaN or
infinite) and isolation distance is NaN for 493 units in 82 files; two isolation distances are above
1e27.

## electrodes-table

> The electrodes table gives each electrode's brain area, MNI coordinates, impedance, the filter applied before spike sorting (300 to 3000 Hz) and the original hardware channel.

Criterion: reproduced when every file has these columns and every electrode has a value for each;
partially when some are empty for some electrodes; not reproduced when a column is empty for every
electrode.

In [ ]:
el = pd.concat([s["electrodes"].assign(identifier=s["identifier"]) for s in sessions], ignore_index=True)
print("Columns over all files:", dict(collections.Counter(c for s in sessions for c in s["electrode_columns"])))
for c in ["location", "x", "y", "z", "imp", "filtering", "origChannel"]:
    v = el[c]
    empty = v.isna() | (v.astype(str).str.strip().isin(["", "nan", "NaN"]))
    print(f"{c}: value for {int((~empty).sum())} of {len(el)} electrodes; files with an empty value: {el[empty].identifier.nunique()}")
print("imp, stored type per file:", dict(collections.Counter(s["imp_dtype"] for s in sessions)),
      "; distinct stored values over all files:", pd.unique(el.imp.astype(str))[:10].tolist())
print("filtering values:", dict(collections.Counter(el.filtering)))
print("coordinate ranges (mm):", {c: (float(np.nanmin(el[c])), float(np.nanmax(el[c]))) for c in "xyz"})

**Result: Not reproduced.** The impedance column is empty for all 887 electrodes: it is stored as float64
and its only value is NaN. Location,
filter (300-3000Hz) and original channel are given for every electrode, and the MNI coordinates for
788 of 887.

## isi-criterion

> Clusters were accepted as units only when fewer than 3% of their interspike intervals were shorter than 3 ms.

Criterion: per unit, the fraction of interspike intervals below 3 ms from its spike times sorted in
time; reproduced when every unit is below 3%, partially when at least 95% are. The spike times as
stored are a variant.

In [ ]:
isi = pd.DataFrame([{"identifier": s["identifier"], "unit": i, "spikes": len(sp), "distinct": len(np.unique(sp)),
                     "unsorted": bool((np.diff(sp) < 0).any()),
                     "sorted": nomem.isi_violation_fraction(sp), "stored": nomem.isi_violation_fraction(sp, sort=False),
                     "distinct_sorted": nomem.isi_violation_fraction(np.unique(sp))}
                    for s in sessions for i, sp in enumerate(s["spikes"])])
print("Units stored out of time order:", int(isi.unsorted.sum()), "; units holding repeated spike times:",
      isi[isi.spikes > isi.distinct][["identifier", "unit", "spikes", "distinct"]].to_dict("records"))
for c in ("sorted", "stored", "distinct_sorted"):
    v = isi[c] * 100
    print(f"{c}: below 3% for {int((v < 3).sum())} of {len(v)} units ({(v < 3).mean() * 100:.1f}%); "
          f"median {v.median():.2f}%, max {v.max():.2f}%; units with fewer than 2 spikes: {int(v.isna().sum())}")
print("Units at 3% or more (sorted):")
print(isi[isi.sorted >= 0.03].sort_values("sorted", ascending=False).round(4).to_string(index=False))

**Result: Partially reproduced.** With spike times sorted, 1858 of 1864 units (99.7%) have fewer
than 3% of intervals below 3 ms; six do not. Two of them (H27_58 units 35 and 37) hold each spike time
twice, stored as two copies one after the other, which gives 50% short intervals once sorted; with
each spike time counted once they are below 3%, and four units (3.0% to 9.0%) remain above.

## fig5a-mtl-locations

> All neurons were recorded from the medial temporal lobe, in the amygdala and the hippocampus.

Criterion: reproduced when the electrode location of every unit names the amygdala or the hippocampus.

In [ ]:
locs = collections.Counter()
for s in sessions:
    loc = s["electrodes"].location.to_numpy()
    for e in s["unit_electrodes"]:
        locs[loc[e[0]]] += 1
print("Units by electrode location:", dict(locs))
mtl = sum(v for k, v in locs.items() if "amygdala" in k.lower() or "hippocampus" in k.lower())
print("Units in amygdala or hippocampus:", mtl, "of", sum(locs.values()))

**Result: Reproduced.** All 1864 units are on electrodes located in the left or right amygdala or
hippocampus.

## Behaviour (Figure 5b to 5d)

Per session, from the 100 recognition trials: the cumulative ROC over the six confidence levels, the
AUC and the z-ROC slope, computed as in the paper's code (`nomem.roc`, `nomem.auc`, `nomem.zroc_slope`).

In [ ]:
rows = []
for s in sessions:
    rec = nomem.recog(s)
    conf, old = rec.confidence.to_numpy(), rec.label.to_numpy() == 1
    h, f = nomem.roc(conf, old)
    a = nomem.auc(h, f)
    fx, fl = nomem.accuracy_fixed(conf, old)
    dx, dl, mode = nomem.accuracy_dynamic(conf, old)
    label, sid = nomem.session_key(s)
    rows.append({"identifier": s["identifier"], "site": nomem.site(label), "auc": a,
                 "auc_stated_coding": nomem.auc(*nomem.roc(conf, ~old)),
                 "zroc_slope": nomem.zroc_slope(h, f) if len(set(np.round(f[:5], 9))) > 1 else np.nan,
                 "distinct_zF": len(set(np.round(f[:5], 9))), "H": h, "F": f,
                 "acc_high_fixed": fx, "acc_low_fixed": fl, "acc_high_dyn": dx, "acc_low_dyn": dl, "split_mode": mode,
                 "included": nomem.included(conf, a), "responses_used": len(set(conf))})
beh = pd.DataFrame(rows)
print("Sessions:", len(beh), "by site:", beh.site.value_counts().to_dict())
print("AUC range:", beh.auc.min().round(3), "to", beh.auc.max().round(3), "; sessions below 0.5:", int((beh.auc < 0.5).sum()))

## fig5c-auc-all

> Subjects distinguished new from old images; the behavioural AUC averaged over all sessions is 0.74 ± 0.10.

Criterion: AUC per session as in the paper's code over all 87 sessions; mean within 0.74 ± 0.10 and
standard deviation within 0.10 ± 0.01.

In [ ]:
m, sd = beh.auc.mean(), beh.auc.std(ddof=1)
print(f"Mean AUC {m:.4f} (0.74 +- 0.10: {abs(m - 0.74) <= 0.10}); SD {sd:.4f} (0.10 +- 0.01: {abs(sd - 0.10) <= 0.01}); n = {len(beh)}")
print(f"Against chance (0.5): t = {stats.ttest_1samp(beh.auc, 0.5).statistic:.2f}, p = {stats.ttest_1samp(beh.auc, 0.5).pvalue:.2g}")
print(f"With the labels as the paper's text states them (0 = old): mean AUC {beh.auc_stated_coding.mean():.4f}")
print(f"Duplicated recognition data: none (sessions 54 and 55 share the encoding block only).")
fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
for _, r in beh.iterrows():
    ax[0].plot(np.r_[0, r.F, 1], np.r_[0, r.H, 1], color="0.8", lw=0.7)
ax[0].plot(np.r_[0, np.mean(np.stack(beh.F), 0), 1], np.r_[0, np.mean(np.stack(beh.H), 0), 1], "o-", color="C0")
ax[0].plot([0, 1], [0, 1], "k:")
ax[0].set(xlabel="false alarm rate", ylabel="hit rate", title="ROC per session (grey) and mean")
ax[1].hist([beh.auc[beh.site == "CS"], beh.auc[beh.site == "TWH"]], bins=np.arange(0.4, 1.01, 0.025), stacked=True, label=["CS", "TWH"])
ax[1].set(xlabel="AUC", ylabel="sessions", title="AUC per session")
ax[1].legend()
plt.tight_layout()
plt.show()
print("All plotted values lie in [0, 1]; AUC range", beh.auc.min().round(3), "to", beh.auc.max().round(3))

**Result: Reproduced.** The mean AUC over the 87 sessions is 0.753 with a standard deviation of 0.094,
both within tolerance of 0.74 ± 0.10.

## fig5c-auc-sites

> Performance was similar at the two study sites, with an average AUC of 0.74 ± 0.09 for CS sessions and 0.69 ± 0.13 for TWH sessions.

Criterion: means within 0.74 ± 0.09 (CS) and 0.69 ± 0.13 (TWH), standard deviations within 0.09 ±
0.009 and 0.13 ± 0.013; partially reproduced when both means are but a spread is not. Welch's t-test
between sites is reported.

In [ ]:
g = beh.groupby("site").auc.agg(["mean", "std", "count"])
print(g.round(4))
checks = {"mean CS": abs(g.loc["CS", "mean"] - 0.74) <= 0.09, "mean TWH": abs(g.loc["TWH", "mean"] - 0.69) <= 0.13,
          "SD CS": abs(g.loc["CS", "std"] - 0.09) <= 0.009, "SD TWH": abs(g.loc["TWH", "std"] - 0.13) <= 0.013}
print(checks)
w = stats.ttest_ind(beh.auc[beh.site == "CS"], beh.auc[beh.site == "TWH"], equal_var=False)
print(f"Welch's t-test CS against TWH: t = {w.statistic:.2f}, p = {w.pvalue:.3f}")
print("TWH AUCs:", beh[beh.site == "TWH"].set_index("identifier").auc.round(3).to_dict())

**Result: Reproduced.** CS sessions have a mean AUC of 0.760 (SD 0.085, n = 78) and TWH sessions 0.688
(SD 0.138, n = 9); all four values are within tolerance. The difference between sites is not
significant (Welch's t-test p = 0.16).

## fig5b-roc-asymmetric

> The ROC curve is asymmetric (p < 0.05), as expected for declarative memory.

Criterion: the slope of the z-ROC per session (regression of z(H) on z(F) over the five interior
points, calcZROCSlope); reproduced when the mean slope is below 1 with a two-sided one-sample t-test
against 1 at p < 0.05. The Wilcoxon signed-rank test and the slope of the pooled ROC are variants.

In [ ]:
sl = beh.zroc_slope.dropna()
print(f"Sessions with a slope: {len(sl)} of {len(beh)}; left out (fewer than 2 distinct false alarm rates): "
      f"{beh[beh.zroc_slope.isna()].identifier.tolist()}")
t = stats.ttest_1samp(sl, 1.0)
print(f"Mean slope {sl.mean():.3f} (SD {sl.std():.3f}, median {sl.median():.3f}); t-test against 1: t = {t.statistic:.2f}, p = {t.pvalue:.2g}")
wx = stats.wilcoxon(sl - 1)
print(f"Variant, Wilcoxon signed-rank against 1: p = {wx.pvalue:.2g}; sessions with slope < 1: {int((sl < 1).sum())} of {len(sl)}")
# pooled ROC: summed counts over sessions (countersAv in NWB_behaviorSummary.m)
conf_all, old_all = [], []
for s in sessions:
    rec = nomem.recog(s)
    conf_all.append(rec.confidence.to_numpy()); old_all.append(rec.label.to_numpy() == 1)
hp, fp = nomem.roc(np.concatenate(conf_all), np.concatenate(old_all))
print(f"Variant, slope of the pooled ROC: {nomem.zroc_slope(hp, fp):.3f}")
# the five sessions left out, with the slope MATLAB's regress returns for a constant z(F)
beh["zroc_slope_regress"] = [nomem.zroc_slope_regress(h, f) for h, f in zip(beh.H, beh.F)]
print("Left-out sessions, slope as regress in calcZROCSlope.m returns it:",
      beh[beh.zroc_slope.isna()].set_index("identifier").zroc_slope_regress.round(3).to_dict())
t_all = stats.ttest_1samp(beh.zroc_slope_regress, 1.0)
print(f"Variant, all 87 sessions with those slopes: mean {beh.zroc_slope_regress.mean():.3f}, t = {t_all.statistic:.2f}, p = {t_all.pvalue:.2g}")
t_one = stats.ttest_1samp(beh.zroc_slope.fillna(1.0), 1.0)
print(f"Variant, all 87 sessions with the left-out slopes set to 1: mean {beh.zroc_slope.fillna(1.0).mean():.3f}, p = {t_one.pvalue:.2g}")
print("Slope range:", sl.min().round(3), "to", sl.max().round(3))

**Result: Reproduced.** Over the 82 sessions with a defined slope the mean z-ROC slope is 0.78, below 1
(t-test against 1, p = 4.3e-09); 65 of 82 slopes are below 1, and the Wilcoxon test (p = 2e-08) and the
pooled ROC (slope 0.82) agree. Five sessions have fewer than two distinct false alarm rates among the
five points, so no regression slope; leaving them out is an assumption of this attempt. The paper's
code would run regress on them, which returns a slope of 0 for four and -0.415 for one; with those
slopes all 87 sessions give a mean of 0.732 (p = 5.4e-10), and with the five set to 1, 0.794 (p =
5.4e-09).

## fig5d-confidence-accuracy

> Accuracy is significantly higher in high than in low confidence trials (P < 0.001, paired t test) across all sessions.

Criterion: per session, the percentage correct among high and among low confidence responses, under
(a) the fixed split of the MATLAB code (high 1 or 6, low 2 to 5) and (b) the dynamic split of the
Python code. Sessions without responses in one group are left out and counted. Reproduced when, under
one reading, high exceeds low with a paired t-test p < 0.001. The unpaired test the code runs, and the
inclusion rule plotFigure5.ipynb applies before this panel (helper.check_inclusion), are variants.

In [ ]:
for name, hi, lo in (("(a) fixed split", "acc_high_fixed", "acc_low_fixed"), ("(b) dynamic split", "acc_high_dyn", "acc_low_dyn")):
    d = beh[[hi, lo, "included", "site"]].dropna(subset=[hi, lo])
    t = stats.ttest_rel(d[hi], d[lo])
    u = stats.ttest_ind(d[hi], d[lo], equal_var=False)
    di = d[d.included]
    ti = stats.ttest_rel(di[hi], di[lo])
    print(f"{name}: {len(d)} sessions ({len(beh) - len(d)} left out); high {100 * d[hi].mean():.1f}%, low {100 * d[lo].mean():.1f}%; "
          f"paired t = {t.statistic:.2f}, p = {t.pvalue:.2g}; high > low in {int((d[hi] > d[lo]).sum())} sessions")
    print(f"    unpaired (Welch) p = {u.pvalue:.2g}; with plotFigure5's inclusion rule, {len(di)} sessions, paired p = {ti.pvalue:.2g}")
    for st, e in d.groupby("site"):
        print(f"    {st}: {len(e)} sessions, high {100 * e[hi].mean():.1f}%, low {100 * e[lo].mean():.1f}%, paired p = {stats.ttest_rel(e[hi], e[lo]).pvalue:.2g}")
print("Split modes chosen by (b):", beh.split_mode.value_counts().to_dict())

**Result: Reproduced.** Under both readings accuracy is higher for high than for low confidence (fixed
split 80.1% against 60.9% over 84 sessions, paired p = 3.2e-16; dynamic split 78.2% against 60.8% over
82 sessions, p = 1.3e-11). The effect holds in the CS sessions alone; in the 8 TWH sessions it has the
same direction without significance (p = 0.14 and 0.26).

## Single units: VS and MS selection

For every unit, the firing rate of each recognition trial in the Methods' window (0.2 to 1.2 s after
onset) and in the code's (0.2 to 1.7 s). VS: one-way ANOVA across the 5 categories, p < 0.05. MS:
bootstrap comparison of mean rates (1000 runs, `nomem.bootstrap_p`) between new and old trials,
p < 0.05, with all 50 and 50 trials (main) or correct trials only (the code's choice). A unit whose rates
are all equal has no ANOVA p and is counted as not selected. The seed is a standard choice; the spread of
the MS count over five more seeds is shown.

In [ ]:
rng = np.random.default_rng(20200304)
rows = []
for s in sessions:
    rec = nomem.recog(s)
    label, sid = nomem.session_key(s)
    on, cat = rec.onset.to_numpy(), rec.category.to_numpy()
    old, conf = rec.label.to_numpy() == 1, rec.confidence.to_numpy()
    correct = (old & (conf >= 4)) | (~old & (conf <= 3))
    for i, sp in enumerate(s["spikes"]):
        row = {"identifier": s["identifier"], "session": sid, "site": nomem.site(label), "unit": i,
               "duplicate": (s["identifier"], i) in duplicate_second}
        for wn, w in (("m", nomem.WINDOW_METHODS), ("c", nomem.WINDOW_CODE)):
            rt = nomem.window_rates(sp, on, w)
            row[f"vs_p_{wn}"] = nomem.anova_p(rt, cat)
            row[f"ms_p_{wn}"] = nomem.bootstrap_p(rt[~old], rt[old], rng)
            row[f"ms_p_{wn}_correct"] = nomem.bootstrap_p(rt[~old & correct], rt[old & correct], rng)
        rows.append(row)
cells = pd.DataFrame(rows)
for c in [c for c in cells if c.endswith(("_m", "_c", "_correct"))]:
    cells[c.replace("_p_", "_")] = cells[c] < nomem.ALPHA
print("Units:", len(cells), "; without an ANOVA p (all rates equal): Methods' window",
      int(cells.vs_p_m.isna().sum()), ", code's window", int(cells.vs_p_c.isna().sum()))
print("Recognition trials per unit: 100 in every session; correct trials per session from",
      min(int(((nomem.recog(s).label == 1) & (nomem.recog(s).confidence >= 4) | (nomem.recog(s).label == 0) & (nomem.recog(s).confidence <= 3)).sum()) for s in sessions),
      "to", max(int(((nomem.recog(s).label == 1) & (nomem.recog(s).confidence >= 4) | (nomem.recog(s).label == 0) & (nomem.recog(s).confidence <= 3)).sum()) for s in sessions))

In [ ]:
# The units selected in each session (index in the file's units table, with its p-value), under the
# main tests (Methods' window, all trials) and the code's choices (0.2 to 1.7 s, MS on correct trials).
def listed(d, sel, pcol):
    return " ".join(f"{u}({p:.1g})" for u, p in zip(d.unit[d[sel]], d[pcol][d[sel]])) or "-"


for ident, d in cells.groupby("identifier", sort=False):
    print(f"{ident}: VS main {listed(d, 'vs_m', 'vs_p_m')} | VS code {listed(d, 'vs_c', 'vs_p_c')} | "
          f"MS main {listed(d, 'ms_m', 'ms_p_m')} | MS code {listed(d, 'ms_c_correct', 'ms_p_c_correct')}")


## vs-cells-proportion

> 343 of the 1863 cells (18.4%) are visually selective, by a one-way ANOVA (P < 0.05) of the firing rate in recognition trials, in a 1 s window starting 200 ms after stimulus onset, across the 5 visual categories.

Criterion: every unit, Methods' window; reproduced when the count is within 343 ± 34.3 and the
percentage within 18.4 ± 1.84. Variants: the code's window, duplicated spike trains counted once, and a
null with the category labels shuffled within each session.

In [ ]:
def show(mask, total, name):
    print(f"{name}: {int(mask.sum())} of {total} ({100 * mask.sum() / total:.1f}%)")

n = len(cells)
show(cells.vs_m, n, "VS, Methods' window, every unit")
print("   within 343 +- 34.3:", abs(cells.vs_m.sum() - 343) <= 34.3, "; within 18.4 +- 1.84%:", abs(100 * cells.vs_m.mean() - 18.4) <= 1.84)
show(cells.vs_c, n, "VS, code's window")
once = cells[~cells.duplicate]
show(once.vs_m, len(once), "VS, Methods' window, duplicated spike trains once")
rng_null = np.random.default_rng(343)
null = 0
for s in sessions:
    rec = nomem.recog(s)
    cat = rng_null.permutation(rec.category.to_numpy())
    for sp in s["spikes"]:
        p = nomem.anova_p(nomem.window_rates(sp, rec.onset.to_numpy(), nomem.WINDOW_METHODS), cat)
        null += int(np.isfinite(p) and p < 0.05)
print(f"Null, categories shuffled within session: {null} of {n} ({100 * null / n:.1f}%); expected by chance 5%")
print("VS by site (Methods' window):", cells.groupby("site").vs_m.agg(["sum", "count"]).to_dict())
print("VS percentage by session: median", (100 * cells.groupby("identifier").vs_m.mean()).median().round(1),
      "; range", (100 * cells.groupby("identifier").vs_m.mean()).min().round(1), "to", (100 * cells.groupby("identifier").vs_m.mean()).max().round(1))

**Result: Reproduced.** 332 of 1864 units (17.8%) are VS in the Methods' window, within tolerance of 343
(18.4%); the code's window gives 343 (18.4%), counting duplicated spike trains once 321 of 1798 (17.9%),
and the shuffled null 76 (4.1%).

## ms-cells-proportion

> 146 of the 1863 cells (7.8%) are memory selective, by a two-tailed bootstrap comparison of mean firing rates (1,000 runs, P < 0.05) between the 50 novel and 50 familiar recognition trials in a 1 s window starting 200 ms after stimulus onset.

Criterion: every unit, Methods' window and all 50 and 50 trials, bootstrap of bootstrapMeanEqualTest at
a fixed seed; reproduced when the count is within 146 ± 14.6 and the percentage within 7.8 ± 0.78.
Variants: the code's choices (correct trials only, its window), duplicated spike trains once, and a null
with labels shuffled within each session.

In [ ]:
show(cells.ms_m, n, "MS, Methods' window, all trials (main)")
print("   within 146 +- 14.6:", abs(cells.ms_m.sum() - 146) <= 14.6, "; within 7.8 +- 0.78%:", abs(100 * cells.ms_m.mean() - 7.8) <= 0.78)
show(cells.ms_m_correct, n, "MS, Methods' window, correct trials")
show(cells.ms_c, n, "MS, code's window, all trials")
show(cells.ms_c_correct, n, "MS, code's window, correct trials (the code's choices)")
show(once.ms_m, len(once), "MS, main, duplicated spike trains once")
# bootstrap noise: the main count with five other seeds
counts = []
for seed in range(1, 6):
    r2 = np.random.default_rng(seed)
    k = 0
    for s in sessions:
        rec = nomem.recog(s)
        old = rec.label.to_numpy() == 1
        for sp in s["spikes"]:
            rt = nomem.window_rates(sp, rec.onset.to_numpy(), nomem.WINDOW_METHODS)
            k += int(nomem.bootstrap_p(rt[~old], rt[old], r2) < 0.05)
    counts.append(k)
print("Main count with seeds 1 to 5:", counts)
rng_null = np.random.default_rng(146)
null = 0
for s in sessions:
    rec = nomem.recog(s)
    old = rng_null.permutation(rec.label.to_numpy() == 1)
    for sp in s["spikes"]:
        rt = nomem.window_rates(sp, rec.onset.to_numpy(), nomem.WINDOW_METHODS)
        null += int(nomem.bootstrap_p(rt[~old], rt[old], rng_null) < 0.05)
print(f"Null, labels shuffled within session: {null} of {n} ({100 * null / n:.1f}%)")
print("MS by site (main):", cells.groupby("site").ms_m.agg(["sum", "count"]).to_dict())

**Result: Not reproduced.** With all 50 and 50 trials in the Methods' window, 118 of 1864 units (6.3%)
are MS, below the tolerance of 146 ± 14.6 (7.8 ± 0.78%); five other seeds give 108 to 115. The
shuffled null gives 92 (4.9%). The code's choices, correct trials only in its 0.2 to 1.7 s window, give
146 (7.8%), the paper's numbers; correct trials alone in the Methods' window give 139 (7.5%), also
within tolerance.

## dual-cells-proportion

> 26 cells (1.4%) are both visually and memory selective.

Criterion: units selected by both main tests; reproduced when the count is within 26 ± 2.6 and the
percentage within 1.4 ± 0.14.

In [ ]:
dual = cells.vs_m & cells.ms_m
show(dual, n, "VS and MS, main tests")
print("   within 26 +- 2.6:", abs(dual.sum() - 26) <= 2.6, "; within 1.4 +- 0.14%:", abs(100 * dual.mean() - 1.4) <= 0.14)
show(cells.vs_c & cells.ms_c_correct, n, "Variant, the code's choices for both")
show(cells.vs_m & cells.ms_m_correct, n, "Variant, Methods' window, MS on correct trials")
exp = cells.vs_m.mean() * cells.ms_m.mean() * n
print(f"Expected if VS and MS were independent: {exp:.1f}")
by_subject = cells.assign(label=cells.identifier.map(info.set_index("identifier").label), dual=dual).groupby("label").agg(
    sessions=("identifier", "nunique"), units=("unit", "size"), VS=("vs_m", "sum"), MS=("ms_m", "sum"), dual=("dual", "sum"))
print("By subject (main tests); subjects with no VS unit:", int((by_subject.VS == 0).sum()), "; with no MS unit:", int((by_subject.MS == 0).sum()))
print(by_subject.T.to_string())

**Result: Not reproduced.** 13 units (0.7%) are both VS and MS under the main tests, half the paper's
26 and below the 21 expected if the two were independent. The code's choices give 25 (1.3%), within
tolerance.

## fig5f-site-proportions

> The percentages of VS, MS and dual cells are comparable between CS sessions (18.4%, 7.7% and 1.4%) and the new TWH sessions (19.4%, 10.4% and 1.3%).

Criterion: two-sided Fisher's exact test between sites for each of VS, MS and dual cells at 0.05;
reproduced when no test finds a difference and at least four of the six values are within tolerance;
partially when only one of these holds. Main tests; the code's choices are a variant.

In [ ]:
expected = {("VS", "CS"): 18.4, ("MS", "CS"): 7.7, ("dual", "CS"): 1.4, ("VS", "TWH"): 19.4, ("MS", "TWH"): 10.4, ("dual", "TWH"): 1.3}


def site_table(vs, ms, label):
    sets = {"VS": vs, "MS": ms, "dual": vs & ms}
    within, fisher = 0, []
    for kind, mask in sets.items():
        a = [int(mask[cells.site == st].sum()) for st in ("CS", "TWH")]
        tot = [int((cells.site == st).sum()) for st in ("CS", "TWH")]
        p = stats.fisher_exact([[a[0], tot[0] - a[0]], [a[1], tot[1] - a[1]]]).pvalue
        fisher.append(p)
        for st, k, t in zip(("CS", "TWH"), a, tot):
            pct = 100 * k / t
            ok = abs(pct - expected[(kind, st)]) <= 0.1 * expected[(kind, st)]
            within += ok
            print(f"  {label} {kind} {st}: {k} of {t} = {pct:.1f}% (paper {expected[(kind, st)]}%; within 10%: {ok})")
        print(f"  {label} {kind}: Fisher's exact p = {p:.3f}")
    print(f"  {label}: {within} of 6 within tolerance; tests finding a difference: {sum(p < 0.05 for p in fisher)}")


site_table(cells.vs_m, cells.ms_m, "main")
site_table(cells.vs_c, cells.ms_c_correct, "code")
print("TWH units by session:", cells[cells.site == "TWH"].groupby("identifier").size().to_dict())

**Result: Partially reproduced.** Under the main tests no Fisher's test finds a difference between CS
(1787 units) and TWH (77 units; p = 0.36, 0.15 and 0.42), but only 3 of the 6 percentages are within
tolerance (CS VS 17.6%, TWH MS 10.4% and TWH dual 1.3%). With the code's choices all 6 are within
tolerance and no test finds a difference. With 77 TWH units the tests have
little power, so finding no difference says little about whether the sites differ.

## twh-sessions

> The dataset includes 9 sessions from a new institution, Toronto Western Hospital.

Criterion: sessions whose subject label starts with T; reproduced when the count is within 9 ± 0.9.

In [ ]:
twh = info[info.label.str.startswith("T")]
print("Sessions with a T label:", len(twh), "; subjects:", twh.label.nunique(), "; institution field:", set(twh.institution))
print(twh[["identifier", "subject_id", "start", "units"]].to_string(index=False))

**Result: Reproduced.** Nine sessions, of eight subjects, have T labels, and their institution field
is Toronto Western Hospital.

## previous-release-link

> The sessions of the earlier release (Faraut et al. 2018) are found in this dandiset under the same subject, session id and task variant.

Link claim, with Table 1 of Faraut et al. 2018 (read from its full text). Criterion: sessions matched
by session id; a matched session agrees when its subject label and task variant are those of the
earlier table. Reproduced when every session of the earlier table is found and agrees; partially when
at least 90% are found and agree, with the others listed. Age and sex of the matched subjects are
compared. The variant of a file is read from its category names (see the data check).

In [ ]:
far_t1, far_totals, far_tables = nomem.faraut_table1()
far = nomem.sessions_of(far_t1)
print("Faraut et al. 2018 Table 1:", len(far_t1), "subjects,", len(far), "sessions; totals row:", far_totals)
print("Faraut et al. 2018 Table 1 as parsed, one row per session:")
print(far.merge(far_t1[["label", "age", "sex"]], on="label").to_string(index=False))
files_by_session = info.set_index("session")
rows = []
for r in far.itertuples():
    found = r.session in files_by_session.index
    f = files_by_session.loc[r.session] if found else None
    rows.append({"session": r.session, "label_2018": r.label, "variant_2018": r.variant, "found": found,
                 "label_file": f.label if found else None, "variant_file": f.variant if found else None,
                 "agrees": bool(found and f.label == r.label and f.variant == r.variant)})
link = pd.DataFrame(rows)
print(f"Found by session id: {int(link.found.sum())} of {len(link)}; found and agree: {int(link.agrees.sum())} "
      f"({100 * link.agrees.mean():.1f}%)")
print("Not found or not agreeing:")
print(link[~link.agrees].to_string(index=False))
ages = far_t1.set_index("label")[["age", "sex"]].join(
    paper_t1.set_index("label")[["age", "sex"]].rename(columns={"age": "age_2020", "sex": "sex_2020"}), how="inner")
subj_file = info.drop_duplicates("label").set_index("label")[["age", "sex"]].rename(columns={"age": "age_file", "sex": "sex_file"})
ages = ages.join(subj_file, how="left")
ages["age_file"] = pd.to_numeric(ages.age_file)
print(f"\nSubjects of the 2018 table also in the files: {int(ages.age_file.notna().sum())} of {len(far_t1)}")
print("Age and sex, 2018 table against the files: both match for",
      int(((ages.age == ages.age_file) & (ages.sex == ages.sex_file)).sum()), "of", int(ages.age_file.notna().sum()))
print(ages[(ages.age != ages.age_file) | (ages.sex != ages.sex_file)].to_string())

**Result: Partially reproduced.** 64 of the 65 sessions of the 2018 table are in the dandiset under the
same subject and variant (98.5%). Session 98 (H48, variant 2) is in neither the dandiset nor Online-only
Table 1. Of the 42 subjects, 39 have the same age and sex in the 2018 table and the files; the 2018
table agrees with Online-only Table 1 on the three others (C24, H33, H48).

## previous-release-additions

> Relative to the earlier release of this task's data (Faraut et al. 2018), this release adds 23 sessions, 17 subjects and 288 neurons.

Criterion: new sessions are those of the dandiset not in the 2018 list, new subjects those with no
session in it, and new neurons the unit rows of the new sessions. Reproduced when all three are within
tolerance (23 ± 2.3, 17 ± 1.7, 288 ± 28.8); partially when one or two are. The differences of the
totals the two papers state are a variant.

In [ ]:
old_sessions, old_labels = set(far.session), set(far.label)
new = info[~info.session.isin(old_sessions)]
new_subjects = sorted(set(info.label) - old_labels)
new_units = int(new.units.sum())
print(f"New sessions: {len(new)} (23 +- 2.3: {abs(len(new) - 23) <= 2.3})")
print(f"New subjects: {len(new_subjects)} (17 +- 1.7: {abs(len(new_subjects) - 17) <= 1.7}): {new_subjects}")
print(f"New neurons (unit rows of new sessions): {new_units} (288 +- 28.8: {abs(new_units - 288) <= 28.8})")
print("New sessions by site:", new.site.value_counts().to_dict())
print(new[["identifier", "units"]].to_string(index=False))
print(f"Variant, differences of totals: sessions 87 - 65 = 22, subjects 59 - 42 = 17, neurons 1863 - 1576 = 287 "
      f"(Faraut et al. 2018 abstract); here {len(info)} - {len(far)} = {len(info) - len(far)} files, "
      f"{info.label.nunique()} - {len(far_t1)} = {info.label.nunique() - len(far_t1)} subjects")

**Result: Partially reproduced.** The dandiset has 23 sessions and 17 subjects that the 2018 table does
not list, as the paper says, but these sessions hold 325 unit rows, above the tolerance of 288 ± 28.8.
The difference of the totals the two papers state (1863 - 1576 = 287) is the paper's number.

## fig4-example-vs-neuron

> Neuron 132-177-4-RA (session 132, channel 177, cell 4, right amygdala) is visually selective, responding most to images of animals.

Criterion: the unit on channel 177 of session 132 with electrode row 4, in the right amygdala;
reproduced when the one-way ANOVA across the 5 categories of recognition trials, Methods' window, has
p < 0.05 and the animal category has the highest mean rate. The code's window is a variant.

In [ ]:
def example(session, channel, row, area):
    s = [x for x in sessions if nomem.session_key(x)[1] == session]
    if not s:
        print(f"session {session} is not in the dandiset")
        return
    s = s[0]
    el = s["electrodes"]
    found = [i for i, e in enumerate(s["unit_electrodes"]) if el.origChannel.iloc[e[0]] == channel]
    print(f"{s['identifier']}: units on channel {channel}: {found}; their electrode rows "
          f"{[int(s['unit_electrodes'][i][0]) for i in found]}; location {set(el.location.iloc[[s['unit_electrodes'][i][0] for i in found]])}")
    rec = nomem.recog(s)
    for i in found:
        for wn, w in (("Methods", nomem.WINDOW_METHODS), ("code", nomem.WINDOW_CODE)):
            rt = nomem.window_rates(s["spikes"][i], rec.onset.to_numpy(), w)
            means = pd.Series(rt).groupby(rec.category_name.to_numpy()).mean()
            print(f"  unit {i} (electrode row {int(s['unit_electrodes'][i][0])}), {wn} window: ANOVA p = {nomem.anova_p(rt, rec.category.to_numpy()):.3g}; "
                  f"mean rate by category (Hz): {means.round(2).to_dict()}; highest: {means.idxmax()}")


example(132, 177, 4, "Right Amygdala")

**Result: Reproduced.** Session 132 has one unit on channel 177, unit 4 on electrode row 4 in the right
amygdala; its rates differ across categories (ANOVA p = 1.2e-22) and are highest for animals (6.75 Hz
against 0.25 to 0.85 Hz), in both windows.

## fig3-example-vs-neuron

> Neuron 144-140-3-RH (session 144, channel 140, cell 3, right hippocampus) is visually selective for phones.

Blocked: there is no session 144 in the dandiset or in Online-only Table 1. The only units on a
channel 140 in the dandiset are in session 2009 (T103), in the right hippocampus; they are shown here,
but no identifier links them to the figure's neuron, so they do not decide the claim.

In [ ]:
example(144, 140, 3, "Right Hippocampus")
print("Sessions with a unit on channel 140:", [s["identifier"] for s in sessions
      if any(s["electrodes"].origChannel.iloc[e[0]] == 140 for e in s["unit_electrodes"])])
example(2009, 140, 3, "Right Hippocampus")

Session 2009 (T103) holds three units; the third (unit 2, on channel 140 in the right hippocampus) is
category selective (ANOVA p = 1.3e-09) with its highest rate for phones (9.1 Hz against 1.7 to 2.7 Hz).
It fits the figure's channel, cell number (counted from 1), area and preferred category, but not its
session id, so the claim stays blocked and the question goes to the paper's authors.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and
the differences from the paper's method.

| Claim | Outcome |
|---|---|
| counts-neurons-subjects | Reproduced: 1864 unit rows (1798 with duplicated spike trains once), 59 subjects |
| counts-sessions | Reproduced: the 87 (subject, session) pairs equal Online-only Table 1 |
| table1-demographics | Partially reproduced: age or sex differ for 3 of 59 subjects |
| task-structure | Partially reproduced: 8 sessions have 50 or 97 encoding trials |
| new-old-label-coding | Not reproduced: label 1 is old and 0 new, the reverse of the stated coding |
| ttl-codes | Partially reproduced: 9 files hold unlisted codes |
| experiment-ids | Partially reproduced: 4 files use id 82 |
| session-dates | Partially reproduced: one date on the 13th, one year 2000 |
| units-table | Partially reproduced: SNR or isolation distance missing for many units |
| electrodes-table | Not reproduced: impedance empty for every electrode |
| isi-criterion | Partially reproduced: 1858 of 1864 units below 3% |
| fig5a-mtl-locations | Reproduced: every unit in amygdala or hippocampus |
| fig5c-auc-all | Reproduced: AUC 0.753 ± 0.094 |
| fig5c-auc-sites | Reproduced: CS 0.760 ± 0.085, TWH 0.688 ± 0.138 |
| fig5b-roc-asymmetric | Reproduced: mean z-ROC slope 0.78, p = 4.3e-09 |
| fig5d-confidence-accuracy | Reproduced: paired p = 3.2e-16 (fixed split) |
| vs-cells-proportion | Reproduced: 332 (17.8%) |
| ms-cells-proportion | Not reproduced: 118 (6.3%); the code's choices give 146 (7.8%) |
| dual-cells-proportion | Not reproduced: 13 (0.7%); the code's choices give 25 (1.3%) |
| fig5f-site-proportions | Partially reproduced: no site difference, 3 of 6 values within tolerance |
| twh-sessions | Reproduced: 9 sessions |
| previous-release-link | Partially reproduced: 64 of 65 earlier sessions found and agree |
| previous-release-additions | Partially reproduced: 23 sessions and 17 subjects new, but 325 units |
| fig4-example-vs-neuron | Reproduced: ANOVA p = 1.2e-22, animals highest |
| fig3-example-vs-neuron | Blocked: session 144 is in neither the dandiset nor Online-only Table 1 |

How the analysis differs from the paper's. The selection of VS and MS cells follows the Methods
(a 1 s window from 200 ms, all 50 new and 50 old trials), as the claims file prescribes; the paper's
code uses 0.2 to 1.7 s and, for MS cells, correct trials only, and those choices give the paper's 343,
146 and 26 (25 here). The bootstrap is seeded. Sessions without a regression slope for the z-ROC (5, an assumption of
this attempt; with the slopes the paper's code would return, p = 5.4e-10) and without responses in a
confidence group (3 or 5) are left out of those tests. Online-only Table 1 and Faraut et
al. 2018 Table 1 are read from Europe PMC.

What was found about the dataset. The new/old labels are coded 1 = old, the reverse of the paper's
text and the column's description. Seven sessions hold only 50 of their encoding trials and one holds
97, so 21 to 24 old images of those sessions were never shown in the file's encoding block. 66 spike
trains are deposited in two files each (sessions 38/39, 43/44 and 54/55), and 54/55 share the encoding
trials. 13 units store spike times out of order, two of them with every spike twice. Electrode
impedance is empty, SNR or isolation distance missing or infinite for many units, ages or sex differ
from the paper's table for three subjects, four files use experiment id 82, nine hold event codes
Table 1 does not list, one start date is not on the 1st and one has the year 2000.

In [ ]:
print(f"Notebook ran in {(time.time() - started) / 60:.1f} min")